# AI-Generated Image Detection — Local Transfer Learning Baseline

## Problem

Binary classification of product-listing images as AI-generated or authentic.

**Task:** given an image, predict:
- `1` → AI-generated
- `0` → Authentic

**Metric:** F1 score on a held-out test set.

---

**Data (local workspace, not included in this repository):**
- a labelled training table and an unlabelled test table, each keyed by image id
- a folder of image files

---

This notebook uses a pretrained CNN to extract image embeddings, then trains a strong classifier on those embeddings.

## 0. Install Dependencies

In [ ]:
!pip install -q torch torchvision timm lightgbm scikit-learn matplotlib tqdm pillow

## 1. Setup Local Paths and Imports

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, f1_score, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
import lightgbm as lgb

ROOT = Path().resolve()
DATA_DIR = ROOT
IMAGES_DIR = DATA_DIR / "images"
TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"

print(f"Workspace root : {ROOT}")
print(f"Images folder  : {IMAGES_DIR}")
print(f"Train CSV      : {TRAIN_CSV}")
print(f"Test CSV       : {TEST_CSV}")
print(f"Total image files found: {len(list(IMAGES_DIR.iterdir()))}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Load Metadata and Verify Images

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

def image_path(image_id):
    path = IMAGES_DIR / image_id
    return str(path) if path.exists() else None

train_df["path"] = train_df["image_id"].apply(image_path)
test_df["path"] = test_df["image_id"].apply(image_path)

missing_train = train_df[train_df["path"].isna()]
missing_test = test_df[test_df["path"].isna()]
print(f"Missing train images: {len(missing_train)}")
print(f"Missing test images : {len(missing_test)}")
if len(missing_train) > 0:
    print(missing_train.head())
if len(missing_test) > 0:
    print(missing_test.head())
    print("Missing test images will be assigned default predictions in the submission.")

train_df = train_df[train_df["path"].notna()].reset_index(drop=True)
test_df_available = test_df[test_df["path"].notna()].copy()

print("Train shape:", train_df.shape)
print("Available train samples:", len(train_df))
print("Test shape :", test_df.shape)
print("Available test samples:", len(test_df_available))
print("Label distribution:")
print(train_df["ground_truth"].value_counts().rename({0: "authentic", 1: "ai_generated"}))

train_df.sample(3, random_state=42)

## 3. Train / Validation Split

In [ ]:
train_split, val_split = train_test_split(
    train_df,
    test_size=0.20,
    stratify=train_df["ground_truth"],
    random_state=42,
)

print(f"Train samples : {len(train_split)}")
print(f"Val samples   : {len(val_split)}")
print("Train class balance:")
print(train_split["ground_truth"].value_counts(normalize=True))
print("Val class balance:")
print(val_split["ground_truth"].value_counts(normalize=True))

## 4. Build a Pretrained Feature Extractor

In [ ]:
IMAGE_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class ImageDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.loc[idx, "path"]
        image = Image.open(path).convert("RGB")
        image = self.transform(image)
        label = self.df.loc[idx, "ground_truth"] if "ground_truth" in self.df.columns else -1
        return image, int(label)


def get_backbone():
    try:
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        backbone = models.resnet50(weights=weights)
    except Exception:
        backbone = models.resnet50(pretrained=True)

    layers = list(backbone.children())[:-1]
    backbone = nn.Sequential(*layers)
    backbone.eval()
    backbone.to(DEVICE)
    return backbone

backbone = get_backbone()
print(backbone)

## 5. Extract Image Embeddings

In [ ]:
@torch.no_grad()
def extract_embeddings(df, batch_size=BATCH_SIZE):
    dataset = ImageDataset(df, transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    embeddings = []

    for images, _ in tqdm(loader, desc="Extracting embeddings"):
        images = images.to(DEVICE)
        output = backbone(images)
        output = output.view(output.size(0), -1)
        embeddings.append(output.cpu().numpy())

    return np.vstack(embeddings)

X_train = extract_embeddings(train_split)
X_val = extract_embeddings(val_split)
X_test = extract_embeddings(test_df_available)

y_train = train_split["ground_truth"].values

y_val = val_split["ground_truth"].values

print("Train embeddings shape:", X_train.shape)
print("Val embeddings shape  :", X_val.shape)
print("Test embeddings shape :", X_test.shape)

## 6. Train a Strong Classifier with Cross-Validation

In [ ]:
N_FOLDS = 5
kf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

X_full = np.vstack([X_train, X_val])
y_full = np.concatenate([y_train, y_val])

oof_preds = np.zeros(len(X_full))
test_preds = np.zeros(len(X_test))
trained_models = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_full, y_full), start=1):
    print(f"\nFold {fold}/{N_FOLDS}")
    X_tr, X_vl = X_full[train_idx], X_full[val_idx]
    y_tr, y_vl = y_full[train_idx], y_full[val_idx]

    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.05,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_vl, y_vl)],
        eval_metric="binary_logloss",
        callbacks=[
            lgb.early_stopping(stopping_rounds=50),
            lgb.log_evaluation(period=50),
        ],
    )

    trained_trained_models.append(model)
    oof_preds[val_idx] = model.predict_proba(X_vl)[:, 1]
    test_preds += model.predict_proba(X_test)[:, 1] / N_FOLDS

print("\nCross-validation complete.")

## 7. Optimize Threshold and Evaluate

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_full, oof_preds)
f1_scores = 2 * precision * recall / (precision + recall + 1e-12)
best_idx = np.nanargmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Best validation threshold by F1: {best_threshold:.4f}")
print(f"validation F1        : {f1_scores[best_idx]:.4f}")

val_preds = (oof_preds > best_threshold).astype(int)
print("\nOverall validation classification report")
print(classification_report(y_full, val_preds, target_names=["Authentic", "AI-Generated"]))

cm = confusion_matrix(y_full, val_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Authentic", "AI-Generated"])
disp.plot(colorbar=False)
plt.title("Validation Confusion Matrix")
plt.tight_layout()
plt.show()

## 8. Inference and Submission

In [ ]:
submission = test_df.copy()
submission["prediction"] = 0

test_df_available["prediction"] = (test_preds > best_threshold).astype(int)
submission.loc[test_df_available.index, "prediction"] = test_df_available["prediction"].values

print("Prediction distribution:")
print(submission["prediction"].value_counts().rename({0: "authentic", 1: "ai_generated"}))

submission[["image_id", "prediction"]].to_csv("submission.csv", index=False)
print("Saved submission.csv")
submission.head()

## 9. Optional: Fine-Tune the Backbone and Ensemble

This section shows how to fine-tune a lightweight ResNet50 head and combine its predictions with the LightGBM ensemble. It is optional but useful if we have GPU time.

In [ ]:
from torchvision import models
# Freeze most of the backbone and fine-tune layer4 + classifier head
def get_finetune_model():
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)
    for name, param in model.named_parameters():
        param.requires_grad = "layer4" in name or "fc" in name

    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),
        nn.Linear(256, 1),
    )
    return model.to(DEVICE)

model_ft = get_finetune_model()

train_dataset = ImageDataset(train_split, transform)
val_dataset = ImageDataset(val_split, transform)
test_dataset = ImageDataset(test_df_available, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model_ft.parameters()), lr=1e-4, weight_decay=1e-5)
class_weights = torch.tensor([(train_split["ground_truth"] == 0).sum() / (train_split["ground_truth"] == 1).sum()], device=DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=class_weights)

@torch.no_grad()
def predict_probs(loader, model):
    model.eval()
    probs = []
    for images, _ in loader:
        images = images.to(DEVICE)
        logits = model(images).view(-1)
        probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)

best_val_loss = float("inf")
for epoch in range(1, 6):
    model_ft.train()
    train_losses = []
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch} train"):
        images = images.to(DEVICE)
        labels = labels.float().to(DEVICE)
        optimizer.zero_grad()
        logits = model_ft(images).view(-1)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    val_losses = []
    model_ft.eval()
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.float().to(DEVICE)
        with torch.no_grad():
            logits = model_ft(images).view(-1)
            loss = criterion(logits, labels)
        val_losses.append(loss.item())

    print(f"Epoch {epoch}: train_loss={np.mean(train_losses):.4f}, val_loss={np.mean(val_losses):.4f}")
    if np.mean(val_losses) < best_val_loss:
        best_val_loss = np.mean(val_losses)
        torch.save(model_ft.state_dict(), "finetuned_resnet50.pt")

model_ft.load_state_dict(torch.load("finetuned_resnet50.pt"))

val_probs_ft = predict_probs(val_loader, model_ft)
test_probs_ft = predict_probs(test_loader, model_ft)

val_preds_ft = (val_probs_ft > 0.5).astype(int)
print("Fine-tuned validation F1:", f1_score(val_split["ground_truth"].values, val_preds_ft))

ensemble_probs = 0.6 * test_preds + 0.4 * test_probs_ft
ensemble_preds = (ensemble_probs > best_threshold).astype(int)

ensemble_submission = test_df.copy()
ensemble_submission["prediction"] = 0
ensemble_submission.loc[test_df_available.index, "prediction"] = ensemble_preds
ensemble_submission[["image_id", "prediction"]].to_csv("submission_ensemble.csv", index=False)
print("Saved submission_ensemble.csv")
print(ensemble_submission["prediction"].value_counts().rename({0: "authentic", 1: "ai_generated"}))

## 10. Notes

- This pipeline uses a pretrained ResNet50 backbone to convert images into dense embeddings.
- Training on embeddings with LightGBM is often faster and more stable than end-to-end fine-tuning on a small dataset.
- The optional fine-tuning section shows how to adapt the backbone and ensemble its output with LightGBM predictions.
- If we want stronger performance, we try additional backbones, model ensembles, or more advanced augmentation.

## 11. ADVANCED: End-to-End Fine-Tuning with Layer-Wise Learning Rates

Advanced fine-tuning strategy with mixed precision, gradient accumulation, and discriminative learning rates

In [ ]:
from torch.cuda.amp import autocast, GradScaler
import warnings
warnings.filterwarnings('ignore')

# ADVANCED FINE-TUNING: Layer-wise learning rates with mixed precision
print("\n" + "="*70)
print("ADVANCED FINE-TUNING: End-to-End ResNet50 with Layer-Wise LR")
print("="*70 + "\n")

def get_advanced_finetune_model():
    """Advanced fine-tuning model with MC Dropout for uncertainty"""
    weights = models.ResNet50_Weights.IMAGENET1K_V2
    model = models.resnet50(weights=weights)

    # Unfreeze layer4 progressively
    for param in model.layer1.parameters():
        param.requires_grad = False
    for param in model.layer2.parameters():
        param.requires_grad = False
    for param in model.layer3.parameters():
        param.requires_grad = True  # Fine-tune layer3
    for param in model.layer4.parameters():
        param.requires_grad = True  # Fine-tune layer4

    # Replace head with advanced architecture
    model.fc = nn.Sequential(
        nn.Dropout(0.5),  # MC Dropout for uncertainty
        nn.Linear(model.fc.in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),  # MC Dropout
        nn.Linear(512, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.3),  # MC Dropout
        nn.Linear(256, 1),
    )

    return model.to(DEVICE)

model_adv = get_advanced_finetune_model()

# Layer-wise learning rates (discriminative learning rates)
def get_layer_wise_optimizer(model, base_lr=1e-4):
    """Create optimizer with different LR for different layers"""
    layer_groups = [
        {'params': model.layer3.parameters(), 'lr': base_lr * 0.1},
        {'params': model.layer4.parameters(), 'lr': base_lr * 0.5},
        {'params': model.fc.parameters(), 'lr': base_lr * 10},
    ]

    optimizer = torch.optim.AdamW(layer_groups, weight_decay=1e-5)
    return optimizer

optimizer_adv = get_layer_wise_optimizer(model_adv, base_lr=1e-4)
criterion_adv = nn.BCEWithLogitsLoss(pos_weight=class_weights)
scaler = GradScaler()  # Mixed precision training

# Training with mixed precision and gradient accumulation
print("Training advanced model with mixed precision and layer-wise LR...\n")

best_val_loss_adv = float("inf")
accumulation_steps = 4

for epoch in range(1, 8):
    model_adv.train()
    train_losses = []

    for step, (images, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}")):
        images = images.to(DEVICE)
        labels = labels.float().to(DEVICE)

        # Mixed precision forward pass
        with autocast():
            logits = model_adv(images).view(-1)
            loss = criterion_adv(logits, labels) / accumulation_steps

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()

        # Gradient accumulation
        if (step + 1) % accumulation_steps == 0:
            scaler.step(optimizer_adv)
            scaler.update()
            optimizer_adv.zero_grad()

        train_losses.append(loss.item() * accumulation_steps)

    # Validation
    model_adv.eval()
    val_losses_adv = []
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.float().to(DEVICE)
        with torch.no_grad():
            with autocast():
                logits = model_adv(images).view(-1)
                loss = criterion_adv(logits, labels)
        val_losses_adv.append(loss.item())

    avg_train_loss = np.mean(train_losses)
    avg_val_loss = np.mean(val_losses_adv)

    print(f"Epoch {epoch}: train_loss={avg_train_loss:.5f}, val_loss={avg_val_loss:.5f}")

    if avg_val_loss < best_val_loss_adv:
        best_val_loss_adv = avg_val_loss
        torch.save(model_adv.state_dict(), "finetuned_resnet50_advanced.pt")
        print(f"  ✓ Improved! Saved checkpoint")

model_adv.load_state_dict(torch.load("finetuned_resnet50_advanced.pt"))

# Get predictions
@torch.no_grad()
def predict_probs_adv(loader, model):
    model.eval()
    probs = []
    for images, _ in loader:
        images = images.to(DEVICE)
        logits = model(images).view(-1)
        probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)

val_probs_adv = predict_probs_adv(val_loader, model_adv)
test_probs_adv = predict_probs_adv(test_loader, model_adv)

val_preds_adv = (val_probs_adv > 0.5).astype(int)
f1_adv = f1_score(val_split["ground_truth"].values, val_preds_adv)
print(f"\n✓ Advanced fine-tuned validation F1: {f1_adv:.4f}")


## 12. ADVANCED: Multi-Modal Ensemble with ViT, EfficientNet, DenseNet

We combine predictions from diverse pre-trained architectures for robust classification

In [ ]:
print("\n" + "="*70)
print("MULTI-MODAL ENSEMBLE: ViT, EfficientNet, DenseNet")
print("="*70 + "\n")

def load_multimodal_models():
    """Load diverse pre-trained architectures"""

    # Vision Transformer
    vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    vit.heads.head = nn.Identity()  # Remove classification head
    vit = vit.eval()

    # EfficientNet-B7
    effnet = models.efficientnet_b7(weights=models.EfficientNet_B7_Weights.IMAGENET1K_V1)
    effnet.classifier = nn.Identity()
    effnet = effnet.eval()

    # DenseNet-201
    densenet = models.densenet201(weights=models.DenseNet201_Weights.IMAGENET1K_V1)
    densenet.classifier = nn.Identity()
    densenet = densenet.eval()

    # Inception V3
    inception = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
    inception.fc = nn.Identity()
    inception.AuxLogits.fc = nn.Identity()
    inception = inception.eval()

    return {
        'vit': vit,
        'efficientnet': effnet,
        'densenet': densenet,
        'inception': inception,
    }

multimodal_models = load_multimodal_models()
for name, model in multimodal_models.items():
    model.to(DEVICE)

print(f"Loaded {len(multimodal_models)} architectures: {list(multimodal_models.keys())}\n")

# Extract multi-modal embeddings
@torch.no_grad()
def extract_multimodal_embeddings(loader, models_dict):
    """Extract embeddings from all architectures"""
    embeddings_dict = {name: [] for name in models_dict.keys()}

    for images, _ in tqdm(loader, desc="Extracting multi-modal embeddings"):
        images = images.to(DEVICE)

        for name, model in models_dict.items():
            with autocast():
                if name == 'vit':
                    # ViT special handling
                    x = model._process_input(images)
                    x = model._reshape_and_permute_patches(x)
                    feat = model.encoder(x)
                    feat = feat[:, 0]  # Class token
                else:
                    feat = model(images)
                    if isinstance(feat, tuple):
                        feat = feat[0]
                    if len(feat.shape) > 2:
                        feat = feat.view(feat.size(0), -1)

            embeddings_dict[name].append(feat.cpu().numpy())

    # Concatenate all
    for name in embeddings_dict:
        embeddings_dict[name] = np.vstack(embeddings_dict[name])

    return embeddings_dict

print("Extracting multi-modal embeddings for validation set...")
val_embeddings_multi = extract_multimodal_embeddings(val_loader, multimodal_models)

print("Extracting multi-modal embeddings for test set...")
test_embeddings_multi = extract_multimodal_embeddings(test_loader, multimodal_models)

# Normalize and concatenate
from sklearn.preprocessing import StandardScaler

print("\nNormalizing and combining embeddings...\n")

val_embeddings_normalized = {}
test_embeddings_normalized = {}

for name in val_embeddings_multi.keys():
    scaler = StandardScaler()
    val_embeddings_normalized[name] = scaler.fit_transform(val_embeddings_multi[name])
    test_embeddings_normalized[name] = scaler.transform(test_embeddings_multi[name])

# Concatenate all modalities
X_val_multimodal = np.concatenate(list(val_embeddings_normalized.values()), axis=1)
X_test_multimodal = np.concatenate(list(test_embeddings_normalized.values()), axis=1)

print(f"✓ Multi-modal validation features shape: {X_val_multimodal.shape}")
print(f"✓ Multi-modal test features shape: {X_test_multimodal.shape}")

# Get multi-modal predictions
@torch.no_grad()
def get_multimodal_probs(loader, models_dict):
    """Get predictions from each architecture"""
    probs = {name: [] for name in models_dict.keys()}

    for images, _ in loader:
        images = images.to(DEVICE)

        for name, model in models_dict.items():
            with autocast():
                if name == 'vit':
                    output = model(images)
                else:
                    output = model(images)
                    if isinstance(output, tuple):
                        output = output[0]

            # Assume output logits, apply sigmoid
            prob = torch.sigmoid(output.view(-1)).cpu().numpy() if output.numel() > 1 else torch.sigmoid(output).cpu().numpy()
            probs[name].append(prob)

    for name in probs:
        probs[name] = np.concatenate(probs[name])

    return probs

val_probs_multi = get_multimodal_probs(val_loader, multimodal_models)
test_probs_multi = get_multimodal_probs(test_loader, multimodal_models)

# Ensemble predictions (weighted average)
weights = {'vit': 0.3, 'efficientnet': 0.25, 'densenet': 0.25, 'inception': 0.2}
val_probs_ensemble_multi = np.average(
    [val_probs_multi[name] for name in weights.keys()],
    axis=0,
    weights=list(weights.values())
)
test_probs_ensemble_multi = np.average(
    [test_probs_multi[name] for name in weights.keys()],
    axis=0,
    weights=list(weights.values())
)

val_preds_multi = (val_probs_ensemble_multi > 0.5).astype(int)
f1_multi = f1_score(val_split["ground_truth"].values, val_preds_multi)
print(f" Multi-modal ensemble validation F1: {f1_multi:.4f}\n")
